# SSTW WAM-frame formal reference Colab 入口

该 Notebook 负责单个 modern external baseline 的项目内官方流程闭环: 以同一 prompt / seed / attack runtime comparison unit 为锚点, 执行 source clone / resource build / official run / adapter 转换 / official bundle 落盘, 并将当前 baseline official bundle 打包为阶段 zip。

Notebook 只调用仓库 helper, 不直接手写 records、tables、figures 或 reports。统一 external baseline runner 的 `metric_status: measured_formal` 转写由 `formal_comparison_scoring_colab.ipynb` 在恢复 5 个 official bundle 后集中执行。


In [ ]:
SSTW_WORKFLOW_PROFILE_VALUE = 'validation_scale'  # 可改为 'probe_paper', 'pilot_paper' 或 'full_paper'
# 1. 挂载 Google Drive 并检查 GPU
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import subprocess
import sys
from pathlib import Path

print(subprocess.getoutput('nvidia-smi'))
DRIVE_PROJECT_ROOT = os.environ.get('SSTW_DRIVE_PROJECT_ROOT', '/content/drive/MyDrive/SSTW')
Path(DRIVE_PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
print('drive_project_root =', DRIVE_PROJECT_ROOT)
# 后续 helper 会恢复前置阶段 zip 到 /content 本地 workspace, 避免循环读取 Drive 小文件。
os.environ.setdefault('SSTW_COLAB_STAGE_IO_MODE', 'local_zip')


In [ ]:
# 1.1 可编辑 workflow profile 切换
# 当前 baseline 参考 Notebook 默认面向 validation_scale。
# 如需复用到 probe_paper、pilot_paper 或 full_paper, 只改第一个代码 cell 第一行。
import os

SSTW_WORKFLOW_PROFILE_VALUE = globals().get('SSTW_WORKFLOW_PROFILE_VALUE', 'validation_scale')


if SSTW_WORKFLOW_PROFILE_VALUE.strip():
    os.environ['SSTW_WORKFLOW_PROFILE'] = SSTW_WORKFLOW_PROFILE_VALUE.strip()
    print('SSTW_WORKFLOW_PROFILE =', os.environ['SSTW_WORKFLOW_PROFILE'])
else:
    os.environ.pop('SSTW_WORKFLOW_PROFILE', None)
    print('SSTW_WORKFLOW_PROFILE 未显式设置。')


In [ ]:
# 2. 冷启动获取仓库代码
REPO_URL = os.environ.get('SSTW_REPO_URL', 'https://github.com/RICHAAARC/SSTW.git')
REPO_DIR = os.environ.get('SSTW_REPO_DIR', '/content/SSTW')
REPO_REF = os.environ.get('SSTW_REPO_REF', '').strip()

if not Path(REPO_DIR).exists():
    if not REPO_URL:
        raise RuntimeError('请先设置 SSTW_REPO_URL, 或将仓库上传到 /content/SSTW')
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print('仓库目录已存在, 执行 git pull 以同步远端代码。')
    %cd "$REPO_DIR"
    !git pull --ff-only
%cd "$REPO_DIR"
if REPO_REF:
    !git fetch --all --tags
    !git checkout "$REPO_REF"
!git rev-parse --short HEAD


In [ ]:
# 3. 安装 baseline 参考运行的公共依赖
# baseline-specific requirements 由 official_runtime_closure_requirements.json 登记。
# 该安装步骤只补齐真实运行环境, 不代表 baseline 已经产出 measured_formal 结果。
# Colab 已预装 torch / torchvision, 此处不使用 -U, 也不主动安装 torchvision, 避免触发 CUDA 栈重装。
%pip install imageio imageio-ffmpeg av opencv-python gdown ffmpeg-python
BASELINE_REQUIREMENTS_FILE = Path('configs/external_baselines/requirements/wam_frame.txt')
INSTALL_BASELINE_REQUIREMENTS = os.environ.get('SSTW_INSTALL_BASELINE_REQUIREMENTS', 'true').lower() == 'true'
if INSTALL_BASELINE_REQUIREMENTS and BASELINE_REQUIREMENTS_FILE.exists():
    !python -m pip install -r "$BASELINE_REQUIREMENTS_FILE"
elif INSTALL_BASELINE_REQUIREMENTS:
    raise RuntimeError(f'缺少 baseline requirements 文件: {BASELINE_REQUIREMENTS_FILE}')
else:
    print('跳过 baseline-specific requirements 安装。')
extra_packages = os.environ.get('SSTW_BASELINE_EXTRA_PIP_PACKAGES', '').strip()
if extra_packages:
    !python -m pip install $extra_packages


In [ ]:
# 4. 可选 Hugging Face 认证
try:
    from huggingface_hub import login
except Exception:
    login = None

if os.environ.get('HF_TOKEN') and login is not None:
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF_TOKEN 状态: provided from environment.')
else:
    print('HF_TOKEN 状态: not_provided 或 huggingface_hub 未安装。')


In [ ]:
# 5. 运行 WAM-frame 官方流程与 official bundle 生成
# Notebook 只调用仓库 helper, 不在 cell 中手写 records、tables、figures 或 reports。
# helper 会以同一 prompt / seed / attack 协议为锚点生成 official bundle,
# 然后调用统一 external_baseline runner 转写 available bundle 为 measured_formal records。
BASELINE_ID = 'wam_frame'
NOTEBOOK_ROLE = 'external_baseline_formal_scoring'
from paper_workflow.colab_utils.wam_frame_formal_reference import run_default_wam_frame_formal_reference_plan

result = run_default_wam_frame_formal_reference_plan(repo_root=Path(os.environ.get('SSTW_REPO_DIR', '/content/SSTW')))
print(json.dumps(result, ensure_ascii=False, indent=2))
print('stage_package_publish_result =', json.dumps(result.get('stage_package_publish_result', {}), ensure_ascii=False, indent=2))

followup_status = {item.get('stage_id'): item.get('stage_status') for item in result.get('followup_results', [])}
print('followup_status =', json.dumps(followup_status, ensure_ascii=False, indent=2))

if result.get('formal_reference_decision') != 'PASS':
    raise RuntimeError(f"{BASELINE_ID} formal reference 未完成: {result.get('formal_reference_status')}")


In [ ]:
# 6. 可选治理审计
# 默认关闭, 避免每个 baseline Notebook 重复运行完整测试。正式提交前仍必须本地运行 pytest 与 harness。
if os.environ.get('SSTW_RUN_QUICK_TESTS_AFTER_BASELINE_REFERENCE', 'false').lower() == 'true':
    !pytest -q
    !python tools/harness/run_all_audits.py
